# Automated Structure Preparation
It represents the second phase of the automated molecular docking pipeline. It bridges the gap between raw data acquisition and the actual docking simulation by transforming standard .pdb structures into the highly specific .pdbqt format required by AutoDock Vina. All operations are executed directly within Google Drive to ensure seamless data handoffs between pipeline stages.
## Workflow & Methodology

## Google Drive Integration:
Mounts the user's Google Drive (force_remount=True) to read the raw files generated in Notebook 1 and ensure processed files are permanently saved and synced.
## Protein Purification (Cleaning):
Parses the raw receptor .pdb file line-by-line. It isolates the macromolecule by retaining only standard amino acid records (ATOM) and chain terminators (TER), effectively stripping away crystallographic waters, solvents, and native heteroatoms (HETATM).
## Receptor Preparation (Open Babel):
-Automatically protonates the cleaned protein at physiological pH (7.4).
-Adds polar hydrogens (essential for detecting hydrogen bonds during docking).
-Calculates and assigns Gasteiger partial charges.
Exports the prepared receptor as a .pdbqt file.

## Ligand Batch Preparation (Open Babel):
-Iterates through all 3D ligand .pdb files in the input directory.
-Adds explicit hydrogens and calculates Gasteiger charges for each small molecule.
-Converts and saves them as individual .pdbqt files.

## Dependencies

-Python Libraries: os, glob, subprocess, google.colab.drive
-External CLI Tools: openbabel (Must be installed via apt-get in the Colab environment).



In [2]:
!apt-get update
!apt-get install -y openbabel


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Package

In [3]:
import os
import glob
import subprocess
from google.colab import drive

# --- 1. Mount Google Drive ---
print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# --- 2. Define Paths ---
BASE_DIR = "/content/drive/MyDrive/Docking_Pipeline"
INPUT_DIR = os.path.join(BASE_DIR, "raw_data")
OUTPUT_DIR = os.path.join(BASE_DIR, "prep_data")
os.makedirs(OUTPUT_DIR, exist_ok=True) # Ensure the folder exists in Drive

PDB_ID = "1HSG"

raw_protein = os.path.join(INPUT_DIR, f"{PDB_ID}.pdb")
clean_protein = os.path.join(OUTPUT_DIR, f"{PDB_ID}_clean.pdb")
receptor_pdbqt = os.path.join(OUTPUT_DIR, f"{PDB_ID}_receptor.pdbqt")

print("\n--- Starting Structure Preparation ---")

# --- 3. Clean the Protein ---
def clean_pdb(input_pdb, output_pdb):
    if not os.path.exists(input_pdb):
        print(f"❌ Error: Cannot find {input_pdb} in Drive!")
        return False

    print(f"\nCleaning {os.path.basename(input_pdb)}...")
    with open(input_pdb, 'r') as f_in, open(output_pdb, 'w') as f_out:
        for line in f_in:
            if line.startswith("ATOM") or line.startswith("TER"):
                f_out.write(line)
    print(f"✅ Cleaned protein saved safely to Drive: {output_pdb}")
    return True

# --- 4. Convert to PDBQT using Open Babel ---
def run_obabel(input_file, output_file, arguments):
    # Run the command and capture any errors
    cmd = f"obabel {input_file} -O {output_file} {arguments}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    # Verify the file was actually created in Google Drive
    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        print(f"  ✅ Success: Saved to {output_file}")
    else:
        print(f"  ❌ Error processing {os.path.basename(input_file)}")
        print(f"  Open Babel Error Message: {result.stderr}")

# --- Execute Processing ---
# Clean Protein
protein_cleaned = clean_pdb(raw_protein, clean_protein)

if protein_cleaned:
    print("\nConverting Protein to PDBQT...")
    # -xr = add polar hydrogens, -xc = add Gasteiger charges, -p 7.4 = protonate
    run_obabel(clean_protein, receptor_pdbqt, "-xr -xc -p 7.4")

# Process Ligands
print("\nProcessing Ligands...")
ligand_files = glob.glob(os.path.join(INPUT_DIR, "ligand_*_3D.pdb"))

if not ligand_files:
    print("❌ No ligand files found in the raw_data folder!")
else:
    for lig_pdb in ligand_files:
        base_name = os.path.basename(lig_pdb).replace(".pdb", ".pdbqt")
        lig_pdbqt = os.path.join(OUTPUT_DIR, base_name)

        print(f"\nConverting {os.path.basename(lig_pdb)} to PDBQT...")
        # -h = add hydrogens, -xc = calculate Gasteiger charges
        run_obabel(lig_pdb, lig_pdbqt, "-h -xc")

print("\n--- Notebook 2 Complete ---")
print(f"Please check your Google Drive folder: MyDrive/Docking_Pipeline/prep_data")

Connecting to Google Drive...
Mounted at /content/drive

--- Starting Structure Preparation ---

Cleaning 1HSG.pdb...
✅ Cleaned protein saved safely to Drive: /content/drive/MyDrive/Docking_Pipeline/prep_data/1HSG_clean.pdb

Converting Protein to PDBQT...
  ✅ Success: Saved to /content/drive/MyDrive/Docking_Pipeline/prep_data/1HSG_receptor.pdbqt

Processing Ligands...

Converting ligand_5281034_3D.pdb to PDBQT...
  ✅ Success: Saved to /content/drive/MyDrive/Docking_Pipeline/prep_data/ligand_5281034_3D.pdbqt

Converting ligand_2244_3D.pdb to PDBQT...
  ✅ Success: Saved to /content/drive/MyDrive/Docking_Pipeline/prep_data/ligand_2244_3D.pdbqt

Converting ligand_3672_3D.pdb to PDBQT...
  ✅ Success: Saved to /content/drive/MyDrive/Docking_Pipeline/prep_data/ligand_3672_3D.pdbqt

Converting ligand_5291_3D.pdb to PDBQT...
  ✅ Success: Saved to /content/drive/MyDrive/Docking_Pipeline/prep_data/ligand_5291_3D.pdbqt

Converting ligand_123631_3D.pdb to PDBQT...
  ✅ Success: Saved to /content/dri